# Lab 04: Orchestration Patterns — Sequential, Parallel & Map-Reduce

## 🎯 What We're Building

| Stage | What | What You Learn |
|-------|------|----------------|
| **Stage 1** | Sequential pipeline with LangGraph | Step-by-step workflows, state passing |
| **Stage 2** | Parallel execution (Fan-Out / Fan-In) | Run independent tasks simultaneously, measure speedup |
| **Stage 3** | Map-Reduce over documents | Process bulk data in parallel, synthesize results |
| **Stage 4** | Supervisor multi-agent system | Manager delegates to specialist agents |

> **Prerequisites:** Complete [Lab 01](../lab-01-react-agent/README.md) and [Lab 02](../lab-02-model-routing/README.md). Read the [README.md](README.md) and [Chapter 7: Orchestration](../../education/en/07-orchestration.md).

---

## 🔧 Setup

This lab only needs **Azure OpenAI** (GPT-4.1). No additional Azure services required.

In [ ]:
import os
import time
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import AzureOpenAI

# ──────────────────────────────────────────────────────
# Load Azure connection strings
# ──────────────────────────────────────────────────────
load_dotenv("../.env")

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
)

MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-41")

print(f"✅ Azure OpenAI connected")
print(f"   Chat model: {MODEL}")

---

# 🏗️ STAGE 1: Sequential Pipeline

The simplest orchestration pattern: steps run **one after another**.
Each step uses the output of the previous step.

```
Research ──→ Draft ──→ Review ──→ Final
  3s           2s        2s       = 7s total
```

### Why sequential?
- Step 2 (Draft) **needs** the output of Step 1 (Research)
- Step 3 (Review) **needs** the output of Step 2 (Draft)
- You can't draft before researching, or review before drafting

### We'll build this with LangGraph StateGraph
- Define a **state** that flows through the pipeline
- Each node reads from state, writes back to state
- LangGraph handles the execution order

In [ ]:
# ──────────────────────────────────────────────────────
# STAGE 1: Sequential Pipeline with LangGraph
#
# We build a 3-step content pipeline:
#   Research → Draft → Review
#
# Each step is a LangGraph node that calls the LLM.
# The state carries data between steps.
# ──────────────────────────────────────────────────────

from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# LLM for LangGraph nodes
llm = AzureChatOpenAI(
    azure_deployment=MODEL,
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
)

# ── Define the state that flows through the pipeline ──
class PipelineState(TypedDict):
    topic: str
    research: str
    draft: str
    final: str

# ── Step 1: Research ──
def research_step(state: PipelineState) -> dict:
    print("📚 Step 1: Researching...")
    start = time.time()
    response = llm.invoke(
        f"Research the topic and provide 4-5 key points with specific data: {state['topic']}"
    )
    elapsed = time.time() - start
    print(f"   ✅ Done ({elapsed:.1f}s)")
    return {"research": response.content}

# ── Step 2: Draft ──
def draft_step(state: PipelineState) -> dict:
    print("✍️  Step 2: Drafting article...")
    start = time.time()
    response = llm.invoke(
        f"Write a short article (200 words) based on this research:\n\n{state['research']}"
    )
    elapsed = time.time() - start
    print(f"   ✅ Done ({elapsed:.1f}s)")
    return {"draft": response.content}

# ── Step 3: Review & Polish ──
def review_step(state: PipelineState) -> dict:
    print("🔍 Step 3: Reviewing & polishing...")
    start = time.time()
    response = llm.invoke(
        f"Review and polish this article. Fix any issues and produce the final version:\n\n{state['draft']}"
    )
    elapsed = time.time() - start
    print(f"   ✅ Done ({elapsed:.1f}s)")
    return {"final": response.content}

# ── Build the LangGraph ──
workflow = StateGraph(PipelineState)
workflow.add_node("research", research_step)
workflow.add_node("draft", draft_step)
workflow.add_node("review", review_step)

workflow.add_edge(START, "research")
workflow.add_edge("research", "draft")
workflow.add_edge("draft", "review")
workflow.add_edge("review", END)

sequential_pipeline = workflow.compile()

# ── Run it! ──
print("📊 STAGE 1: Sequential Pipeline")
print("=" * 60)
print("   research → draft → review (each waits for the previous)\n")

overall_start = time.time()
result = sequential_pipeline.invoke({
    "topic": "The impact of AI agents on software engineering in 2026"
})
total_time = time.time() - overall_start

print(f"\n{'─' * 60}")
print(f"📄 Final article preview:")
print(f"{result['final'][:500]}...")
print(f"\n⏱️  Total sequential time: {total_time:.1f}s")
print(f"   Each step waited for the previous one!")

### 🤔 What did you notice?

- Each step took ~2-3 seconds (one LLM call each)
- Total time = sum of all steps
- The draft **used** the research output, the review **used** the draft
- This is correct behavior — these steps genuinely depend on each other

**Sequential is the right pattern when steps have dependencies.**

But what if the steps are **independent**? Then sequential is wasteful...

---

# 🏗️ STAGE 2: Parallel Execution (Fan-Out / Fan-In)

Now imagine we need to search **3 different sources** for the same query.
These searches are **independent** — Source A doesn't need Source B's results.

```
Sequential:                    Parallel:
                                      ┌─ Source A (2s) ─┐
Source A (2s)                         │                  │
  → Source B (2s)        vs    Query ─┤─ Source B (2s) ─├─ Merge
    → Source C (2s)                   │                  │
                                      └─ Source C (2s) ─┘
Total: ~6s                    Total: ~2s (3x faster!)
```

### Fan-Out / Fan-In Pattern
- **Fan-Out (Scatter):** Send the same query to multiple workers
- **Fan-In (Gather):** Collect all results and merge them

### We'll use Python's ThreadPoolExecutor
- Simple, built-in, no extra dependencies
- Perfect for I/O-bound tasks (LLM API calls)
- Each worker runs in its own thread

In [ ]:
# ──────────────────────────────────────────────────────
# STAGE 2: Parallel Execution
#
# We search 3 sources for the same query:
#   1. Sequential (baseline) — one at a time
#   2. Parallel (optimized)  — all at once
#
# Then we compare wall-clock time to see the speedup.
# ──────────────────────────────────────────────────────

def search_source(source_name: str, query: str) -> dict:
    """Search a single source. Each call takes ~1-3 seconds (one LLM call)."""
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": f"You are an expert from {source_name}. Provide 3-4 key findings about the topic. Be concise."},
            {"role": "user", "content": query}
        ],
        max_tokens=200
    )
    elapsed = time.time() - start
    return {
        "source": source_name,
        "content": response.choices[0].message.content,
        "time": elapsed
    }

query = "What are the latest trends in AI agent orchestration?"
sources = ["Academic Research", "Industry News", "Technical Blogs"]

print("📊 STAGE 2: Parallel Execution")
print("=" * 60)

# ── SEQUENTIAL: one source at a time ──
print("\n🐌 Sequential (one at a time):")
seq_start = time.time()
seq_results = []
for source in sources:
    r = search_source(source, query)
    seq_results.append(r)
    print(f"   ✅ {r['source']}: {r['time']:.1f}s")
seq_time = time.time() - seq_start
print(f"   Total: {seq_time:.1f}s")

# ── PARALLEL: all sources at once ──
print(f"\n⚡ Parallel (all at once):")
par_start = time.time()
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(search_source, s, query) for s in sources]
    par_results = [f.result() for f in futures]
for r in par_results:
    print(f"   ✅ {r['source']}: {r['time']:.1f}s")
par_time = time.time() - par_start
print(f"   Total: {par_time:.1f}s")

# ── MERGE: Combine results (Fan-In) ──
print(f"\n🔄 Fan-In: Merging results from all sources...")
combined_context = "\n\n".join(
    f"[{r['source']}]: {r['content']}" for r in par_results
)
merge_start = time.time()
merge_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Synthesize the findings from multiple sources into a unified summary. Highlight where sources agree and where they differ."},
        {"role": "user", "content": combined_context}
    ],
    max_tokens=300
)
merge_time = time.time() - merge_start
print(f"   ✅ Merge: {merge_time:.1f}s")

# ── Compare ──
speedup = seq_time / par_time
print(f"\n{'─' * 60}")
print(f"📊 Comparison:")
print(f"   Sequential: {seq_time:.1f}s (search only)")
print(f"   Parallel:   {par_time:.1f}s (search only)")
print(f"   Speedup:    {speedup:.1f}x faster!")
print(f"\n📄 Merged Summary:")
print(merge_response.choices[0].message.content[:400])
print(f"\n💡 With {len(sources)} independent sources, parallel is ~{len(sources)}x faster!")

### 🤔 What did you notice?

- Sequential took ~6s (3 × ~2s), Parallel took ~2s — roughly **3x speedup**!
- The merge step adds a small overhead but is worth it
- All 3 searches ran **simultaneously** in separate threads
- The total parallel time ≈ the **slowest** single search (not the sum)

### When to use parallel:
- Tasks are **independent** (no dependencies between them)
- You have multiple I/O-bound operations (API calls, database queries)
- You want to reduce wall-clock time

### Pitfalls to watch for:
- **Rate limits** — too many parallel LLM calls can hit API rate limits
- **Cost** — parallel doesn't save money, just time (same number of LLM calls)
- **Merge complexity** — combining results from different sources requires thought

---

# 🏗️ STAGE 3: Map-Reduce

Map-Reduce is a powerful pattern for processing **many items**:

```
┌─────────────────────────────────────────────────────┐
│                                                      │
│  MAP PHASE:                                          │
│  4 documents → 4 parallel LLM calls → 4 summaries   │
│                                                      │
│  REDUCE PHASE:                                       │
│  4 summaries → 1 LLM call → 1 executive summary     │
│                                                      │
└─────────────────────────────────────────────────────┘
```

### Why Map-Reduce?

| Approach | 4 docs | 100 docs |
|----------|--------|----------|
| Sequential | 4 × 2s = 8s | 100 × 2s = 200s (3+ min) |
| Map-Reduce | 2s + 2s = 4s | 2s + 2s = 4s! |

Map-Reduce scales beautifully — 4 docs or 100 docs, same wall-clock time
(assuming enough parallel workers and API capacity).

We'll use the **Acme Corp documents from Lab 03** to demonstrate.

In [ ]:
# ──────────────────────────────────────────────────────
# STAGE 3: Map-Reduce
#
# We'll process the Acme Corp documents from Lab 03:
#   MAP:    Summarize each document in parallel
#   REDUCE: Combine all summaries into one executive brief
#
# Compare sequential vs parallel timing.
# ──────────────────────────────────────────────────────

# Load documents from Lab 03's data directory
data_dir = Path("../lab-03-memory-rag/data")
documents = []
for file_path in sorted(data_dir.glob("*.md")):
    content = file_path.read_text()
    documents.append({
        "filename": file_path.name,
        "content": content
    })
    print(f"📄 Loaded: {file_path.name} ({len(content)} chars)")

print(f"\n✅ Loaded {len(documents)} documents for Map-Reduce")

# ── MAP FUNCTION: Summarize one document ──
def summarize_doc(doc: dict) -> dict:
    """Summarize a single document (one LLM call)."""
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Summarize this document in 2-3 concise sentences. Include the most important details and numbers."},
            {"role": "user", "content": doc["content"]}
        ],
        max_tokens=150
    )
    elapsed = time.time() - start
    return {
        "filename": doc["filename"],
        "summary": response.choices[0].message.content,
        "time": elapsed
    }

print(f"\n{'=' * 60}")
print("📊 STAGE 3: Map-Reduce Pattern")
print("=" * 60)

# ── SEQUENTIAL MAP (baseline) ──
print("\n🐌 Sequential Map (one doc at a time):")
seq_start = time.time()
seq_summaries = []
for doc in documents:
    r = summarize_doc(doc)
    seq_summaries.append(r)
    print(f"   ✅ {r['filename']}: {r['time']:.1f}s")
seq_map_time = time.time() - seq_start
print(f"   Map total: {seq_map_time:.1f}s")

# ── PARALLEL MAP ──
print(f"\n⚡ Parallel Map (all docs at once):")
par_start = time.time()
with ThreadPoolExecutor(max_workers=len(documents)) as executor:
    par_summaries = list(executor.map(summarize_doc, documents))
for r in par_summaries:
    print(f"   ✅ {r['filename']}: {r['time']:.1f}s")
par_map_time = time.time() - par_start
print(f"   Map total: {par_map_time:.1f}s")

# ── REDUCE: Combine all summaries ──
print(f"\n📊 Reduce (combine all summaries into one brief):")
combined = "\n".join(
    f"[{s['filename']}]: {s['summary']}" for s in par_summaries
)
reduce_start = time.time()
reduce_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are given individual document summaries from a company's internal library. Synthesize them into one cohesive executive summary (150 words max) that captures the key points across all documents."},
        {"role": "user", "content": combined}
    ],
    max_tokens=250
)
reduce_time = time.time() - reduce_start
print(f"   ✅ Reduce: {reduce_time:.1f}s")

# ── Results ──
map_speedup = seq_map_time / par_map_time
print(f"\n{'─' * 60}")
print(f"📄 Executive Summary (from {len(documents)} documents):")
print(reduce_response.choices[0].message.content)

print(f"\n{'─' * 60}")
print(f"📊 Map-Reduce Timing:")
print(f"   Sequential map:   {seq_map_time:.1f}s")
print(f"   Parallel map:     {par_map_time:.1f}s  (speedup: {map_speedup:.1f}x)")
print(f"   Reduce:           {reduce_time:.1f}s")
print(f"   Total (parallel):  {par_map_time + reduce_time:.1f}s")
print(f"   Total (sequential): {seq_map_time + reduce_time:.1f}s")
print(f"\n💡 Imagine 100 documents — parallel map still takes ~{par_map_time:.0f}s!")

### 🤔 What did you notice?

- **Map phase** benefited hugely from parallelism (4 docs in the time of 1)
- **Reduce phase** is always sequential (one LLM call to synthesize)
- Total time = max(map times) + reduce time
- This scales to hundreds of documents!

### Map-Reduce in production:

| Application | Map Step | Reduce Step |
|------------|----------|-------------|
| Summarize 100 docs | Summarize each | Synthesize all summaries |
| Analyze 50 support tickets | Classify each | Generate trend report |
| Review 20 code PRs | Review each | Write summary of all changes |
| Translate 30 pages | Translate each | Combine into one document |

### Rate limit considerations:
In production, you'd use a **semaphore** to limit concurrent workers:
```python
# Limit to 10 concurrent LLM calls (respect rate limits)
semaphore = asyncio.Semaphore(10)
```

---

# 🏗️ STAGE 4: Supervisor Multi-Agent System

The most advanced pattern: a **Manager agent** that coordinates **specialist agents**.

```
         ┌─ 🔍 Researcher (finds data) ──────┐
         │                                     │
🎩 Manager ─┤─ 📊 Analyst (analyzes patterns) ───├─ Final deliverable
         │                                     │
         └─ ✍️ Writer (polishes content) ──────┘
```

### How it works:
1. The Manager receives a task from the user
2. The Manager decides which specialist to call first
3. Each specialist does their work and returns results
4. The Manager coordinates the full workflow
5. The Manager produces the final output

### Key insight:
The Manager is itself a **ReAct agent** (from Lab 01). The specialists are
its **tools**. The Manager **reasons** about which specialist to call next,
just like our Lab 01 agent reasoned about which function to call.

In [ ]:
# ──────────────────────────────────────────────────────
# STAGE 4: Supervisor Multi-Agent System
#
# We build a Manager agent with 3 specialist "sub-agents":
#   🔍 Researcher — finds information and data
#   📊 Analyst   — analyzes data, identifies patterns
#   ✍️ Writer    — produces polished deliverables
#
# Each specialist is a tool that the Manager can call.
# The Manager decides who to call and in what order.
# ──────────────────────────────────────────────────────

from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# ── Specialist 1: Researcher ──
@tool
def call_researcher(query: str) -> str:
    """Call the Research Specialist to gather facts, data points, and background
    information about a topic. Use this FIRST to collect raw information."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a research specialist. Provide detailed factual findings with specific data points and statistics. Be thorough but concise. Cite types of sources."},
            {"role": "user", "content": query}
        ],
        max_tokens=400
    )
    return f"[Research Findings]\n{response.choices[0].message.content}"

# ── Specialist 2: Analyst ──
@tool
def call_analyst(data: str) -> str:
    """Call the Analysis Specialist to analyze data and identify patterns,
    trends, and implications. Use this AFTER gathering research."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an analysis specialist. Analyze the provided information, identify key patterns, trends, risks, and opportunities. Provide 3-5 actionable insights."},
            {"role": "user", "content": data}
        ],
        max_tokens=400
    )
    return f"[Analysis]\n{response.choices[0].message.content}"

# ── Specialist 3: Writer ──
@tool
def call_writer(content: str) -> str:
    """Call the Writing Specialist to create a polished, well-structured document.
    Use this LAST to produce the final deliverable."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a professional writer. Create a polished, well-structured executive brief based on the provided content. Use clear headings, bullet points, and concise language. Keep it under 300 words."},
            {"role": "user", "content": content}
        ],
        max_tokens=500
    )
    return f"[Final Document]\n{response.choices[0].message.content}"

# ── Create the Supervisor Agent ──
SUPERVISOR_PROMPT = """You are a project manager coordinating a team of specialists.

Your team:
1. **Researcher** (call_researcher) — finds facts and data
2. **Analyst** (call_analyst) — analyzes data and finds patterns
3. **Writer** (call_writer) — creates polished deliverables

Your workflow:
1. FIRST, call the researcher to gather information
2. THEN, call the analyst to analyze the research findings
3. FINALLY, call the writer to produce the final deliverable

Always follow this order. Pass the output of each step to the next specialist.
After the writer returns, present the final document to the user."""

supervisor = create_react_agent(
    llm,
    [call_researcher, call_analyst, call_writer],
    prompt=SUPERVISOR_PROMPT
)

# ── Run the Supervisor ──
print("📊 STAGE 4: Supervisor Multi-Agent System")
print("=" * 60)
print("   🎩 Manager → 🔍 Researcher → 📊 Analyst → ✍️ Writer")
print("   The manager decides who to call and when.\n")

sup_start = time.time()
result = supervisor.invoke({
    "messages": [("user",
        "Prepare a brief for the CEO on how AI agents are transforming "
        "customer support in 2026. Include current trends, key statistics, "
        "and 3 recommendations for our company."
    )]
})
sup_time = time.time() - sup_start

# Show the workflow trace
print(f"\n📋 Supervisor Workflow Trace:")
for msg in result["messages"]:
    if hasattr(msg, 'type'):
        if msg.type == 'ai' and hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"   🎩 Manager → calls {tc['name']}")
        elif msg.type == 'tool':
            print(f"   📨 {msg.name} returned results ({len(msg.content)} chars)")

# Show final output
final_msg = result["messages"][-1].content
print(f"\n{'─' * 60}")
print(f"📄 Final CEO Brief:")
print(final_msg[:800])
print(f"\n{'─' * 60}")
print(f"⏱️  Total supervisor time: {sup_time:.1f}s")
print(f"💡 The manager coordinated 3 specialists automatically!")

### 🤔 What did you notice?

- The Manager **decided on its own** to call Researcher → Analyst → Writer
- Each specialist has its own system prompt and expertise
- The Manager passed research results to the analyst, and analysis to the writer
- This is a **pipeline**, but the Manager could also decide to skip steps or loop back

### When to use the Supervisor pattern:

| Scenario | Why Supervisor? |
|----------|----------------|
| Complex reports | Different skills needed (research, analysis, writing) |
| Customer support escalation | Route to specialist based on issue type |
| Code review pipeline | Linter → Security check → Style review → Summary |
| Content creation | Research → Draft → Edit → SEO optimize |

### Trade-offs:
- **Pros:** Modular (swap specialists easily), clear separation of concerns
- **Cons:** More LLM calls (manager + each specialist), higher cost and latency

---

---

# 🎓 Summary: What We Built and Learned

### The Four Patterns

| Pattern | When to Use | Speedup | Complexity |
|---------|------------|---------|------------|
| **Sequential** | Steps have dependencies | None (baseline) | ⭐ Low |
| **Parallel** | Independent tasks | ~Nx for N tasks | ⭐⭐ Medium |
| **Map-Reduce** | Bulk processing | ~Nx for N items | ⭐⭐ Medium |
| **Supervisor** | Multi-expertise tasks | Depends on workflow | ⭐⭐⭐ High |

### Key Concepts

| Concept | What It Means |
|---------|---------------|
| **Sequential** | Steps run one after another, each depends on previous |
| **Parallel (Fan-Out/Fan-In)** | Independent tasks run simultaneously, results merged |
| **Map-Reduce** | Split bulk work into parallel subtasks, combine results |
| **Supervisor** | Manager agent delegates to specialist sub-agents |
| **StateGraph** | LangGraph's way to define step-by-step workflows |
| **ThreadPoolExecutor** | Python's built-in parallel execution mechanism |
| **Speedup** | Wall-clock time reduction from parallelism |

### Choosing the Right Pattern

```
Is the task a single step?
  → YES: No orchestration needed (Lab 01 ReAct agent)
  → NO: Do steps depend on each other?
    → YES: Sequential Pipeline (Stage 1)
    → NO: Are all subtasks the same operation?
      → YES: Map-Reduce (Stage 3)
      → NO: Parallel Fan-Out/Fan-In (Stage 2)

Does the task need multiple types of expertise?
  → YES: Supervisor Multi-Agent (Stage 4)
```

### What's Next?

In **Lab 05**, we'll explore **Tools & Safety** — creating custom tools,
adding input validation, DLP scanning, and budget guardrails.

---

> **[← Back to Lab 03](../lab-03-memory-rag/README.md)** | **[→ Lab 05: Tools & Safety](../lab-05-tools-safety/README.md)**